# 02 — Preprocessing QC

Phase 2 quality-control notebook. Loads cleaned epochs from `data/processed/`, compares PSD before/after preprocessing, and visualizes ICA components removed.

**Stop condition:** Clean PSD with 1/f slope, no broadband shelf above 30 Hz on temporal channels (T7/T8/FT7/FT8 — EMG marker), reasonable rejection rate per subject.

In [ ]:
import sys, json
sys.path.insert(0, '..')
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mne
mne.set_log_level('WARNING')

from src.loader import load_raw_continuous, load_subject
from src.preprocessor import load_clean_epochs, preprocess_subject

In [ ]:
# Pick a subject with preprocessed data
processed_dir = Path('../data/processed')
available = sorted([f.stem.replace('-clean-epo', '') for f in processed_dir.glob('*-clean-epo.fif')])
print(f'Preprocessed subjects: {available}')

SUBJECT = available[0] if available else 'MM05'
print(f'Inspecting: {SUBJECT}')

clean_epochs, log = load_clean_epochs(SUBJECT, processed_dir='../data/processed')
print(f'\nLog summary:')
for k in ['bad_channels', 'ica_n_components', 'ica_labels_rejected',
         'n_epochs_before_reject', 'n_epochs_after_reject', 'rejection_rate']:
    print(f'  {k}: {log[k]}')

In [ ]:
# Compare PSD: raw (uncleaned) vs preprocessed
raw_epochs, _ = load_subject(SUBJECT, raw_data_dir='../data/raw')

psd_raw = raw_epochs.compute_psd(method='welch', fmin=0.5, fmax=80, n_fft=2048, n_overlap=512)
psd_clean = clean_epochs.compute_psd(method='welch', fmin=0.5, fmax=80, n_fft=2048, n_overlap=512)

psd_raw_data, freqs = psd_raw.get_data(return_freqs=True)
psd_clean_data, _ = psd_clean.get_data(return_freqs=True)

# Mean across trials and channels (all channels for raw, good channels only for clean)
good_idx = [i for i, ch in enumerate(clean_epochs.ch_names) if ch not in clean_epochs.info['bads']]
raw_mean = psd_raw_data.mean(axis=(0, 1))
clean_mean = psd_clean_data[:, good_idx, :].mean(axis=(0, 1))

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(freqs, raw_mean, label='Raw (pre-preprocessing)', color='gray', lw=1.5, alpha=0.7)
ax.semilogy(freqs, clean_mean, label='Clean (good channels only)', color='steelblue', lw=1.5)
for lo, hi, c, name in [(1,4,'#e8d5b7','δ 1-4'), (4,8,'#b7d5e8','θ 4-8'),
                          (8,12,'#b7e8c8','α 8-12'), (12,30,'#e8b7d5','β 12-30')]:
    ax.axvspan(lo, hi, alpha=0.1, color=c)
ax.axvline(60, color='red', lw=1, ls='--', alpha=0.5, label='60 Hz')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (V²/Hz)')
ax.set_title(f'{SUBJECT} — PSD before vs after preprocessing')
ax.legend()
ax.set_xlim(0, 80)
plt.tight_layout()
plt.savefig(f'../outputs/figures/{SUBJECT}_psd_compare.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Check temporal channels for EMG shelf (broadband above 30 Hz indicates muscle artifact)
temporal_channels = ['T7', 'T8', 'FT7', 'FT8']
available_temporal = [ch for ch in temporal_channels if ch in clean_epochs.ch_names]
print(f'Temporal channels available: {available_temporal}')

fig, axes = plt.subplots(1, len(available_temporal), figsize=(4 * len(available_temporal), 4), sharey=True)
if len(available_temporal) == 1:
    axes = [axes]
for ax, ch in zip(axes, available_temporal):
    ch_idx = clean_epochs.ch_names.index(ch)
    is_bad = ch in clean_epochs.info['bads']
    raw_psd = psd_raw_data[:, ch_idx, :].mean(axis=0)
    clean_psd = psd_clean_data[:, ch_idx, :].mean(axis=0)
    ax.semilogy(freqs, raw_psd, color='gray', alpha=0.6, label='raw', lw=1)
    ax.semilogy(freqs, clean_psd, color='steelblue', label='clean', lw=1.5)
    title = f'{ch} (BAD)' if is_bad else ch
    ax.set_title(title, color='red' if is_bad else 'black')
    ax.set_xlim(0, 80)
    ax.set_xlabel('Hz')
    ax.axvspan(30, 80, alpha=0.1, color='red', label='EMG band')
    ax.legend(fontsize=8)
axes[0].set_ylabel('PSD (V²/Hz)')
fig.suptitle(f'{SUBJECT} — Temporal channels (EMG check)')
plt.tight_layout()
plt.savefig(f'../outputs/figures/{SUBJECT}_emg_check.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Per-epoch peak-to-peak distribution (good channels only)
data = clean_epochs.get_data()
ptp_per_ch_per_epoch = (data[:, good_idx, :].max(axis=2) - data[:, good_idx, :].min(axis=2)) * 1e6  # µV
ptp_per_epoch = ptp_per_ch_per_epoch.max(axis=1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ptp_per_epoch, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(100, color='red', ls='--', label='100 µV')
ax.axvline(np.median(ptp_per_epoch), color='green', label=f'median={np.median(ptp_per_epoch):.0f} µV')
ax.set_xlabel('Max peak-to-peak across good channels (µV)')
ax.set_ylabel('Epochs')
ax.set_title(f'{SUBJECT} — Cleaned epoch amplitude distribution (n={len(clean_epochs)})')
ax.legend()
plt.tight_layout()
plt.savefig(f'../outputs/figures/{SUBJECT}_ptp_dist.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Min: {ptp_per_epoch.min():.0f} µV, Median: {np.median(ptp_per_epoch):.0f} µV, Max: {ptp_per_epoch.max():.0f} µV')

In [ ]:
# Trials per class — check class balance after rejection
classes = sorted(log['trials_per_class'].keys())
counts = [log['trials_per_class'][c] for c in classes]
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(classes, counts, color='steelblue', edgecolor='black')
ax.axhline(15, color='gray', ls='--', label='Original 15/class', alpha=0.5)
ax.axhline(5, color='red', ls='--', label='Min for 5-fold CV', alpha=0.5)
ax.set_ylabel('Trials kept after preprocessing')
ax.set_title(f'{SUBJECT} — Trials per class (kept {sum(counts)}/{log["n_epochs_before_reject"]})')
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, str(n),
             ha='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig(f'../outputs/figures/{SUBJECT}_classes.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Aggregate summary across all preprocessed subjects
all_logs = []
for s in available:
    with open(f'../data/processed/{s}-log.json') as f:
        all_logs.append(json.load(f))

import pandas as pd
df = pd.DataFrame([{
    'subject': L['subject'],
    'bad_channels': L['n_bad_channels'],
    'ica_components': L['ica_n_components'],
    'ica_rejected': len(L['ica_components_rejected']),
    'eye_blinks': L['ica_labels_rejected'].get('eye blink', 0),
    'muscle': L['ica_labels_rejected'].get('muscle artifact', 0),
    'heart': L['ica_labels_rejected'].get('heart beat', 0),
    'trials_in': L['n_epochs_before_reject'],
    'trials_out': L['n_epochs_after_reject'],
    'reject_rate_%': round(L['rejection_rate'] * 100, 1),
    'classes_kept': len(L['trials_per_class']),
    'duration_s': round(L['duration_seconds'], 0),
} for L in all_logs])
print(df.to_string(index=False))